# 06 Matching

Each code cell below is one of the five scripts of `06_matching/`, in order: the same code as the command line (`python 06_matching/NN_name.py`), plus a first line setting `__file__`, which the scripts use to find `output/`. Change a cell to experiment; to change an example for good, edit its script and run `python sync_notebooks.py`, which copies the scripts into the notebooks. Each script's docstring says what it shows, what it needs and what it checks.

SIMBA's matching is a module of its own, `simba.Modules.Matching`. A `.def` names what to vary and what to hit, in a line's `match:` (transverse optics) or `longitudinal_match:` (the RF) block, and SIMBA sets the elements before the line is tracked. A `match:` runs in a code with a matcher: by default the line's own code, or the one named by `code:`; Ocelot, Xsuite, MAD-X, elegant and Bmad use their own matchers, Cheetah SIMBA's least-squares solver. Targets are SIMBA's Twiss names (`beta_x`, `eta_x`, `mux` in units of 2π, ...) and the transfer matrix `R11` to `R66`, in Xsuite's coordinates whatever the code. Variables are quadrupoles, any number on an element (`B1.angle`), or a group's own parameter (a chicane's `angle`); a functional definition is matched, not overwritten.

All five use JFEL from laura-lattices. `jfel.py` holds what they share: the undulator's periodic Twiss (from example 23) and a `.def` from the injector beam through the S-band linac, the X-band linearizer and the VBC chicane.

## 33 The same match in every code

Example 23's match (the S13 quads, from half strength, onto the undulator's periodic Twiss), once per code with `code:`. Every code puts the tracked beam on the target (β within 0.3%). Four quads and four targets have more than one exact solution: Ocelot, MAD-X, Cheetah, elegant and Bmad land on the strengths stored in the lattice (to $6\times10^{-6}$), Xsuite on another that meets the targets as well. A Cheetah line asked for a dispersion target, which Cheetah's optics cannot give, hands over to the first code that can, with a warning saying why.

In [ ]:
__file__ = "33_match_in_any_code.py"  # python and %run set this; a notebook cell does not
"""
33 - The same match in every code
=================================

Shows: where a `match:` block runs. SIMBA's matching is its own module
(simba.Modules.Matching); a `.def` names what to vary and what to hit, and
the match runs in a code with a matcher. By default that is the line's own
code; `code:` in the block picks another:

    match:
      variables: [JFEL-S13-MAG-QUAD-01, ..., -04]
      targets: {JFEL-FEL-SIM-MARK-01: {beta_x: ..., alpha_x: ..., ...}}
      code: xsuite

Ocelot, Xsuite, MAD-X and Bmad use their own matchers (Tao's for Bmad),
elegant its optimiser; Cheetah, which has none, SIMBA's least-squares
solver on Cheetah's optics. Targets are SIMBA's Twiss names whatever the code.

The script runs example 23's match (JFEL's S13 quads, from half strength,
onto the undulator's periodic Twiss), once per code, on the same Ocelot line
and beam. It then shows the default: a Cheetah line with a dispersion target,
which Cheetah's optics cannot give, is matched in the first code that can
(simba.Modules.Matching.transverse.FALLBACK), with a warning that says why.

Run:    python 06_matching/33_match_in_any_code.py
Tier:   A (elegant runs if it is installed or in the container; Bmad if
        pytao and $ACC_ROOT_DIR's libtao.so are; each prints SKIP otherwise)
Needs:  SIMBA, LAURA, laura-lattices (JFEL), Ocelot, Xsuite, MAD-X, Cheetah.
Output: output/33_match_in_any_code/: run/ and periodic/.

Units: k1l in 1/m; beta in m.

Four quads and four targets can have more than one exact solution. From the
same start, Ocelot, MAD-X, Cheetah, elegant and Bmad land on the strengths
stored in the lattice (which came from Ocelot's match) to ~6e-6; Xsuite's
solver lands on another (k1l about 1.28, -1.21, -0.46, 1.44 1/m), which
meets the targets as well. Which solution is wanted (the weaker quads, a
smaller beta inside) is the user's to say, with more targets or bounds.

Checks:
- After each match, the tracked beam at the undulator entrance is on the
  target to 2% in beta and 0.02 in alpha, as in example 23.
- Ocelot's match gives the stored strengths to 1e-3 (measured 6e-6).
- The Cheetah line hands over with a warning, and its match meets the target too.
"""

import warnings
from pathlib import Path

import simba.Framework as fw
from jfel import ENTRANCE, JFEL_DIR, S13, S13_QUADS, beam_twiss, differences, periodic_target

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
RUN = OUT / "run"
CODES = ("ocelot", "xsuite", "madx", "cheetah", "elegant", "bmad")

framework = fw.Framework(master_lattice=JFEL_DIR, directory=str(RUN), clean=True, verbose=False)
framework.loadSettings("Lattices/jfel_match_undulator.def")
for line in ("Linac", "Match", "Undulator"):
    framework[line].csr_enable = False
stored = [framework.getElement(q).k1l for q in S13_QUADS]
framework.track(files=["Linac"], check_lattice=False)
target = periodic_target(OUT, RUN / S13)


def match_in(code=None, extra_targets=None):
    """Match from half strength, in `code` (None: the default), and track; the matched k1l and entrance Twiss."""
    for q, k1l in zip(S13_QUADS, stored):
        framework.modifyElement(q, "k1l", 0.5 * k1l)
    block = framework["Match"].file_block["match"]
    block["targets"] = {ENTRANCE: {**target, **(extra_targets or {})}}
    block.pop("code", None)
    if code:
        block["code"] = code
    framework.track(files=["Match"], check_lattice=False)
    return [framework.getElement(q).k1l for q in S13_QUADS], beam_twiss(RUN / f"{ENTRANCE}.openpmd.hdf5")


results, problems = {}, []
for code in CODES:
    try:
        results[code] = match_in(code)
    except ValueError as error:  # the code cannot run here; says why
        print(f"SKIP {code}: {error}")

# The default: a Cheetah line with a target Cheetah cannot give hands over
framework.change_Lattice_Code("Match", "cheetah")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    handed = match_in(extra_targets={"eta_x": ["a<", 0.01]})
handover = [str(w.message) for w in caught if "cannot match this" in str(w.message)]
print("\n" + (handover[0] if handover else "no hand-over warning"))
results["cheetah line (default)"] = handed

print(f"\n{'k1l [1/m]':24s}" + "".join(f"{q[-7:]:>9s}" for q in S13_QUADS) + "   max |k1l/stored - 1|")
print(f"{'stored':24s}" + "".join(f"{v:9.4f}" for v in stored))
for label, (k1l, _) in results.items():
    worst = max(abs(m / s - 1) for m, s in zip(k1l, stored))
    print(f"{label:24s}" + "".join(f"{v:9.4f}" for v in k1l) + f"   {worst:.1e}")
if max(abs(m / s - 1) for m, s in zip(results["ocelot"][0], stored)) > 1e-3:
    problems.append("Ocelot's match is not the stored strengths")

print(f"\n{'beam at entrance':24s} beta vs target  alpha vs target")
for label, (_, twiss) in results.items():
    beta, alpha = differences(twiss, target)
    print(f"{label:24s} {beta:13.1e}  {alpha:15.4f}")
    if beta > 0.02 or alpha > 0.02:
        problems.append(f"{label} Twiss")

# --- Checks ----------------------------------------------------------------
if not handover:
    problems.append("no hand-over warning")
assert not problems, f"failed: {problems}"
print(f"\nOK: {len(results) - 1} codes match the beam onto the undulator, and the Cheetah line hands over")

## 34 Transfer-matrix targets: point-to-point imaging

`R12 = R34 = 0` from S13 to the undulator entrance. Xsuite, elegant and Bmad match on the matrix themselves; Ocelot's matcher has no matrix targets and MAD-X's refuses more variables than constraints, so SIMBA's solver works on their matrices; Cheetah declines. Each code's solution is checked by tracking: each particle's position at the entrance fitted against its coordinates at S13. The fit must carry the chromatic terms, or it returns the matrix averaged over the energy spread ($R_{12} + U_{1266}\langle\delta^2\rangle$, 0.02 to 0.26 m off here); with them the tracked $R_{12}$ and $R_{34}$ are below $4\times10^{-5}$ m, against 4.1 and 2.8 m unmatched.

In [ ]:
__file__ = "34_matrix_targets.py"  # python and %run set this; a notebook cell does not
"""
34 - Transfer-matrix targets: point-to-point imaging
====================================================

Shows: targets on the transfer matrix. `R11` to `R66` at an element are the
matrix from the start of the line to the end of that element, in Xsuite's
coordinates (x, px, y, py, zeta, delta) whatever the code matches with: each
code's own matrix is converted (Ocelot's tau and MAD-X's t by a sign and a
power of beta0, elegant's path length by its own rule). The block

    match:
      variables: [JFEL-S13-MAG-QUAD-01, ..., -04]
      targets: {JFEL-FEL-SIM-MARK-01: {R12: 0, R34: 0}}

asks for point-to-point imaging from S13 to the undulator entrance: a
particle's position there does not depend on its angle at S13. Xsuite,
elegant and Bmad match on R natively. Ocelot's matcher has no matrix targets,
and MAD-X's MATCH refuses more variables (four quads) than constraints (two),
so for those SIMBA's least-squares solver works on the code's own matrices.
Cheetah takes no matrix targets (SIMBA has no convention for its matrices)
and says so.

Run:    python 06_matching/34_matrix_targets.py
Tier:   A (elegant and Bmad run if installed, as in example 33)
Needs:  SIMBA, LAURA, laura-lattices (JFEL), Ocelot, Xsuite, MAD-X.
Output: output/34_matrix_targets/run/.

Units: R12 and R34 in m (x in m over px in rad); R11, R33 dimensionless.

Two targets on four quads leave a family of solutions, so the codes land on
different strengths (elegant on much stronger quads); each must give imaging.
That is checked by tracking, not with any code's matrix: the Match line is
tracked (in Ocelot, collective effects off) after each match, and each
particle's x and y at the entrance are fitted against its x, px, y, py and
delta at S13. The fit has to carry the chromatic terms (x delta, px delta,
and with delta^2): a fit linear in delta returns the matrix averaged over
the energy spread, R12 + U1266 <delta^2>, which is 0.02 to 0.26 m off here,
most for the strongest quads. With them, the fitted R12 is the on-energy one
the target asks for, and the unmatched one equals the codes' (4.122 m).

Checks:
- After every code's match, the tracked R12 and R34 are below 1e-4 m, against
  4.1 and 2.8 m unmatched (measured: at most 3.5e-5 m, elegant's stopping point;
  1e-5 m or less for the others).
- Cheetah declines, giving its reason.
"""

from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from jfel import ENTRANCE, JFEL_DIR, S13, S13_QUADS

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
RUN = OUT / "run"
CODES = ("ocelot", "xsuite", "madx", "elegant", "bmad", "cheetah")
IMAGING = {ENTRANCE: {"R12": 0.0, "R34": 0.0}}


def tracked_matrix(start, end):
    """
    On-energy R11, R12, R33, R34, fitted from the tracked beam: each particle's x
    and y at the end against its x, px, y, py and delta at the start. The fit
    carries the chromatic terms (x delta, px delta, and their delta^2), or it
    would return the matrix averaged over the energy spread, R12 + U1266 <delta^2>,
    not the R12 the target is for.
    """
    def coordinates(beam):
        return beam.x.val, beam.cpx.val / beam.cpz.val, beam.y.val, beam.cpy.val / beam.cpz.val
    x0, xp0, y0, yp0 = coordinates(start)
    d0 = start.cp.val / np.mean(start.cp.val) - 1
    linear = [x0, xp0, y0, yp0]
    chromatic = [u * d0**n for u in linear for n in (1, 2)]
    a = np.column_stack(linear + [d0, d0**2] + chromatic + [np.ones_like(d0)])
    x1, _, y1, _ = coordinates(end)
    rx = np.linalg.lstsq(a, x1, rcond=None)[0]
    ry = np.linalg.lstsq(a, y1, rcond=None)[0]
    return {"R11": rx[0], "R12": rx[1], "R33": ry[2], "R34": ry[3]}


framework = fw.Framework(master_lattice=JFEL_DIR, directory=str(RUN), clean=True, verbose=False)
framework.loadSettings("Lattices/jfel_match_undulator.def")
for line in ("Linac", "Match"):
    framework[line].csr_enable = False
# the matrix is the lattice's: compare it with a beam that sees only the lattice
framework["Match"].lsc_enable = framework["Match"].wakefield_enable = False
stored = [framework.getElement(q).k1l for q in S13_QUADS]
framework.track(files=["Linac"], check_lattice=False)
start = rbf.load_file(str(RUN / f"{S13}.openpmd.hdf5"))


def match_in(code):
    """Match imaging from half strength in `code`, track, and return the k1l and the tracked R."""
    for q, k1l in zip(S13_QUADS, stored):
        framework.modifyElement(q, "k1l", 0.5 * k1l)
    framework["Match"].file_block["match"].update(targets=IMAGING, code=code)
    framework.track(files=["Match"], check_lattice=False)
    end = rbf.load_file(str(RUN / f"{ENTRANCE}.openpmd.hdf5"))
    return [framework.getElement(q).k1l for q in S13_QUADS], tracked_matrix(start, end)


# Unmatched, at half strength, for scale
framework["Match"].file_block["match"]["enable"] = False
for q, k1l in zip(S13_QUADS, stored):
    framework.modifyElement(q, "k1l", 0.5 * k1l)
framework.track(files=["Match"], check_lattice=False)
unmatched = tracked_matrix(start, rbf.load_file(str(RUN / f"{ENTRANCE}.openpmd.hdf5")))
framework["Match"].file_block["match"]["enable"] = True

results, declined = {}, {}
for code in CODES:
    try:
        results[code] = match_in(code)
    except ValueError as error:
        declined[code] = str(error)
        print(f"SKIP {code}: {error}")

def row(label, k1l, r):
    return (f"{label:10s}" + "".join(f"{v:9.4f}" for v in k1l)
            + f"{r['R11']:9.4f}{r['R12']:10.1e}{r['R33']:9.4f}{r['R34']:10.1e}")


print(f"\n{'':10s}" + "".join(f"{q[-7:]:>9s}" for q in S13_QUADS) + "      R11       R12      R33       R34   (tracked)")
print(row("unmatched", [0.5 * k for k in stored], unmatched))
problems = []
for code, (k1l, r) in results.items():
    print(row(code, k1l, r))
    if abs(r["R12"]) > 1e-4 or abs(r["R34"]) > 1e-4:
        problems.append(code)

# --- Checks ----------------------------------------------------------------
if "cheetah" not in declined:
    problems.append("Cheetah did not decline")
assert not problems, f"failed: {problems}"
print(f"\nOK: {len(results)} codes match point-to-point imaging, confirmed by tracking; Cheetah declines")

## 35 Match a chicane's R56 with its angle

`bunch_compressor.angle` as the variable: the chicane group lays its dipoles out again at every angle tried. From the design 0.13 rad onto the R56 the chicane has at 0.12 rad (example 16's exact formula), in every code: the angle comes back as 0.120000 rad, the tracked R56 is the target to $10^{-5}$, the screen in the middle straight moves where the layout puts it, the dipoles' `angle` edges follow, and the settings' `dipoleangle` keeps the new angle. The Cheetah line hands over.

In [ ]:
__file__ = "35_chicane_r56.py"  # python and %run set this; a notebook cell does not
"""
35 - Match a chicane's R56 with its angle
=========================================

Shows: a group as a match variable. JFEL's VBC chicane is a `groups:`
chicane (example 16); its `angle` is a variable of its own,

    match:
      variables: [bunch_compressor.angle]
      targets: {JFEL-VBC-SIM-MARK-01: {R56: ...}}

and the match sets it through the group: `set_angle` lays the four dipoles
and everything between them out again at every angle tried, so the model is
the chicane as it would be built. A group whose name is given with a field
it does not have (an `element_group`'s members' `k1l`, say) means each member's.

The dipoles' edge angles follow: JFEL writes them as `angle` (dipole 1's exit,
dipole 2's entrance, ...), which LAURA evaluates from each bend's angle when
it writes the lattice. An edge written as a number would stay fixed, and the
match would warn so.

The match sets the angle in the settings' `groups` too (`dipoleangle`), so a
line built later, or the saved settings, keep it.

The script runs the VBC on its own, from a chirp-free 130 MeV beam (as in
example 16), starting at its design 0.13 rad, and matches it to the R56 the
chicane has at 0.12 rad, in each code: by default in the line's own code.
A Cheetah line hands over, as Cheetah takes no matrix targets.

Run:    python 06_matching/35_chicane_r56.py
Tier:   A (elegant and Bmad run if installed, as in example 33)
Needs:  SIMBA, LAURA, laura-lattices (JFEL), Ocelot, Xsuite, MAD-X, Cheetah.
Output: output/35_chicane_r56/: bc.def, start.openpmd.hdf5, one run per code.

Units: angle in rad; R56 in m (z in m, delta = dp/p: Xsuite's zeta and delta).

The target is example 16's exact R56 at 0.12 rad: the path-length R56 of a
rectangular chicane with the arc and gap set_angle gives there, plus the
velocity term S/gamma^2.

Checks, in every code that runs:
- The matched angle is 0.12 rad to 1e-4 (R56 goes as angle^2, so this is
  the target R56 to 2e-4).
- The tracked R56 (example 16's fit) equals the target to 2e-4.
- The screen in the middle straight is where the chicane at that angle puts it.
- Dipole 1's exit edge is still `angle`, and the settings' dipoleangle is the angle.
"""

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import simba.Modules.Beams as rbf

ROOT = Path(__file__).resolve().parents[1]
sys.path[:0] = [str(ROOT / "03_modifying"), str(ROOT / "02_tracking")]
from chicane import DIPOLES, END, SCREEN, bend_geometry, linear_coefficients, load, write_def  # noqa: E402
from simple_lattice import M_E, make_beam  # noqa: E402

OUT = ROOT / "output" / Path(__file__).stem
OUT.mkdir(parents=True, exist_ok=True)
CODES = ("ocelot", "xsuite", "madx", "elegant", "bmad", "cheetah")
TARGET_ANGLE = 0.12
LIBTAO = os.path.join(os.environ.get("ACC_ROOT_DIR", ""), "production", "lib", "libtao.so")

start = make_beam(OUT / "start.openpmd.hdf5", beta=(10.0, 10.0), p0=130e6, sigma_pz=130e3, particles=5000)
gamma = np.mean(start.cp.val) / M_E
def_file = write_def(OUT / "bc.def", OUT / "start")


def exact_r56(framework, angle):
    """Example 16's R56 for the chicane as laid out at `angle` in `framework`."""
    rho, gap = bend_geometry(framework, angle)
    s = framework["BC"].getSValues()
    r56_path = 2 * (2 * rho * (np.tan(angle) - angle) + gap * np.tan(angle) ** 2 / np.cos(angle))
    return r56_path + (s[-1] - s[0]) / gamma**2


target = exact_r56(load(def_file, OUT / "target", "ocelot", angle=TARGET_ANGLE), TARGET_ANGLE)
print(f"target R56 {target:.6f} m (the chicane's at {TARGET_ANGLE} rad)")

rows, problems, handover = [], [], []
for code in CODES:
    if code == "bmad" and not os.path.isfile(LIBTAO):
        print("SKIP bmad: tracking needs $ACC_ROOT_DIR/production/lib/libtao.so (export ACC_ROOT_DIR)")
        continue
    run = OUT / code
    framework = load(def_file, run, code)  # at the design angle, 0.13 rad
    if code == "bmad":
        framework["BC"].libtao = LIBTAO  # as example 32
    framework["BC"].file_block["match"] = {
        "variables": ["bunch_compressor.angle"],
        "targets": {END: {"R56": target}},
    }
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            framework.track()
    except ValueError as error:
        print(f"SKIP {code}: {error}")
        continue
    handover += [str(w.message) for w in caught if "cannot match this" in str(w.message)]
    group = framework["bunch_compressor"]
    angle = group.angle
    end = rbf.load_file(str(run / f"{END}.openpmd.hdf5"))
    r56 = linear_coefficients(start, end.z.val - start.z.val)[2]
    rho, gap = bend_geometry(framework, angle)
    x_screen = framework[SCREEN].physical.middle.x
    x_expected = 2 * rho * (1 - np.cos(angle)) + gap * np.tan(angle)
    kept = framework.settings["groups"]["bunch_compressor"].get("dipoleangle")
    edge = framework[DIPOLES[0]].magnetic.exit_edge_angle
    rows.append((code, angle, r56, x_screen, x_expected, kept, edge))
    if abs(angle / TARGET_ANGLE - 1) > 1e-4:
        problems.append(f"{code}: angle {angle}")
    if abs(r56 / target - 1) > 2e-4:
        problems.append(f"{code}: tracked R56 {r56:.6f}")
    if abs(x_screen - x_expected) > 1e-6:
        problems.append(f"{code}: screen x {x_screen:.6f}, expected {x_expected:.6f}")
    if kept is None or abs(kept - angle) > 1e-12 or edge != "angle":
        problems.append(f"{code}: dipoleangle {kept}, exit edge {edge!r}")

print(f"\n{'line':8s} {'angle':>9s} {'R56 tracked':>12s} {'rel diff':>9s} {'screen x':>9s} {'laid out':>9s} "
      f"{'dipoleangle':>11s} {'D1 exit edge':>12s}")
for code, angle, r56, x, xe, kept, edge in rows:
    print(f"{code:8s} {angle:9.6f} {r56:12.6f} {r56 / target - 1:9.1e} {x:9.6f} {xe:9.6f} {kept:11.6f} {edge!s:>12s}")
print("\n" + (handover[0] if handover else "no hand-over warning"))

# --- Checks ----------------------------------------------------------------
if not handover:
    problems.append("the Cheetah line did not hand over")
assert not problems, "\n".join(problems)
print(f"\nOK: {len(rows)} lines match the chicane's R56 through its angle; the layout, edges and settings follow")

## 36 Set the linac RF for a chirp, linearised by the X-band

`longitudinal_match` (Zagorodnov and Dohlus): L02 and L03, linked by a functional definition `linac_phase`, as the main system and L4H as its fourth harmonic, for a chirp of $-15$ m$^{-1}$ with zero curvature and skewness at the nominal energy. One analytic pass is close (chirp 3% off, skewness cut fifteenfold), not exact: JFEL enters L02 at 35 MeV, where the bunch slips 1.5° against the wave, and the structure wakes take energy from the tail. Closing the loop, as the method is used in practice (track, move the targets by the miss, match again), puts the tracked beam on the targets in three passes without wakes and four with them. Tracked in Xsuite, which applies the RF exactly; `linac_phase` is what changes, so L02 and L03 stay linked.

In [ ]:
__file__ = "36_longitudinal_match.py"  # python and %run set this; a notebook cell does not
"""
36 - Set the linac RF for a chirp, and linearise it with the X-band
==================================================================

Shows: `longitudinal_match`, after Zagorodnov and Dohlus (PRST-AB 14, 014403,
2011). From the line's input beam, it sets a main RF system and a harmonic of
it so the beam leaves with a given energy gain and a given chirp, curvature
and skewness of its relative energy along the bunch:

    longitudinal_match:
      cavities: linac            # a group: JFEL's S-band L02 and L03
      harmonics: [JFEL-L4H-LIN-CAV-01]   # the X-band, 4x the frequency
      chirp: -15.0               # [1/m], the coefficient of s in (E - E0)/E0
      curvature: 0.0             # [1/m^2]: linearised
      skewness: 0.0              # [1/m^3]
      energy_gain: ...           # [eV] over the line; by default what its cavities give now

with s = c(<t> - t), the head at positive s. The equations are linear in
V cos(phi) and V sin(phi) of each system; each system's voltage is shared
between its cavities as they share it now, and all its cavities take one
phase. Travelling-wave cavities (all three here) are set by their gradient,
through LAURA's voltage factor.

L02 and L03 are linked as JFEL's operators might link them: the .def
(jfel.py) sets both phases to a functional definition, `linac_phase`. The
match sets that definition, not the cavities, so they stay linked.

The method's model is a rigid, ultra-relativistic beam and the RF alone.
JFEL enters L02 at 35 MeV, where the bunch slips ~1.5 deg against the wave
over a 4 m structure, and the structure wakes take energy from the tail: so
one analytic pass is close, not exact. The script then closes the loop as
the method is used in practice: track, move the targets by what was missed,
match again. Three passes bring the tracked beam onto the targets without
wakes; with them, which change as the RF reshapes the bunch, four. Tracking is in Xsuite, which applies the RF to each
slice of the cavities exactly (Cheetah's and Ocelot's cavity maps are second
order, so they give the curvature approximately and the skewness not at all).

Run:    python 06_matching/36_longitudinal_match.py
Tier:   A
Needs:  SIMBA, LAURA, laura-lattices (JFEL) and Xsuite.
Output: output/36_longitudinal_match/: nominal/, rf/ and wakes/.

Units: energy in MeV; chirp 1/m, curvature 1/m^2, skewness 1/m^3 (of (E-E0)/E0 in s).

Checks, after the passes, without wakes and with them:
- The tracked chirp is the target to 0.2%, the curvature and skewness are
  below 1% of what the nominal RF leaves (109 1/m^2, 6.3e5 1/m^3 at L4H's
  -184 deg), and the energy at the bunch centre is the nominal one to 1e-4.
- L02 and L03 still read `linac_phase`, which the match changed, and keep
  their nominal gradient ratio (30 : 27 MV/m).
- The first, analytic pass is within 5% in chirp and has cut the skewness
  by more than ten.
"""

from pathlib import Path

import numpy as np
import simba.Framework as fw
from jfel import JFEL_DIR, L02, L03, L4H, VBC_START, compressor_def, longitudinal_shape

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
OUT.mkdir(parents=True, exist_ok=True)
TARGET = np.array([-15.0, 0.0, 0.0])  # chirp, curvature, skewness
NAMES = ("chirp", "curvature", "skewness")
PASSES = 4


def framework_for(name, wakes):
    framework = fw.Framework(master_lattice=JFEL_DIR, directory=str(OUT / name), clean=True, verbose=False)
    framework.loadSettings(str(compressor_def(OUT / f"{name}.def", "xsuite")))
    line = framework["Linac"]
    line.csr_enable = line.lsc_enable = False
    line.wakefield_enable = wakes
    return framework


def track(framework, name):
    framework.track(files=["Linac"])
    return longitudinal_shape(OUT / name / f"{VBC_START}.openpmd.hdf5")


# The nominal RF, without wakes: the energy to keep, and the curvature to beat
e_in = longitudinal_shape(Path(JFEL_DIR) / "Data_Files" / "jfel_injector_S02.openpmd.hdf5")[0]
e_nominal, nominal = track(framework_for("nominal", wakes=False), "nominal")
print(f"nominal: {e_nominal / 1e6:.3f} MeV, " + ", ".join(f"{n} {v:.4g}" for n, v in zip(NAMES, nominal)))

problems, first = [], {}
for name, wakes in (("rf", False), ("wakes", True)):
    framework = framework_for(name, wakes)
    asked, gain = TARGET.copy(), e_nominal - e_in
    print(f"\n{name}: {'pass':>4s} {'E [MeV]':>9s} " + "".join(f"{n:>11s}" for n in NAMES)
          + f" {'linac_phase':>11s} {'L4H phase':>9s}")
    for n in range(1, PASSES + 1):
        framework["Linac"].file_block["longitudinal_match"] = {
            "cavities": "linac", "harmonics": [L4H], "energy_gain": float(gain),
            **dict(zip(NAMES, map(float, asked))),
        }
        energy, shape = track(framework, name)
        if n == 1:
            first[name] = shape
        phase = framework.settings["functional_definitions"]["linac_phase"]
        print(f"{'':6s} {n:4d} {energy / 1e6:9.3f} " + "".join(f"{v:11.4g}" for v in shape)
              + f" {phase:11.3f} {framework.getElement(L4H).cavity.phase:9.3f}")
        asked, gain = asked + (TARGET - shape), gain + (e_nominal - energy)  # move the targets by the miss

    if abs(shape[0] / TARGET[0] - 1) > 2e-3:
        problems.append(f"{name}: chirp {shape[0]:.4f}")
    if abs(shape[1]) > 0.01 * abs(nominal[1]) or abs(shape[2]) > 0.01 * abs(nominal[2]):
        problems.append(f"{name}: curvature {shape[1]:.3f} or skewness {shape[2]:.3e}")
    if abs(energy / e_nominal - 1) > 1e-4:
        problems.append(f"{name}: energy {energy / 1e6:.3f} MeV")
    cavities = [framework.getElement(c) for c in (L02, L03)]
    if any(c.cavity.phase != "linac_phase" for c in cavities):
        problems.append(f"{name}: L02/L03 lost their linac_phase")
    ratio = cavities[0].simulation.field_amplitude / cavities[1].simulation.field_amplitude
    if abs(ratio / (30 / 27) - 1) > 1e-9:
        problems.append(f"{name}: L02:L03 gradient ratio {ratio}")

# --- Checks ----------------------------------------------------------------
if abs(first["rf"][0] / TARGET[0] - 1) > 0.05 or abs(first["rf"][2]) > 0.1 * abs(nominal[2]):
    problems.append(f"first analytic pass: {first['rf']}")
assert not problems, "\n".join(problems)
print(f"\nOK: {PASSES} passes put the beam on the chirp, linearised, at the nominal energy, with and without wakes")

## 37 Set up JFEL's bunch compression by matching

Both matches together, for a compression factor $C = 4$ with a chirp $h = -15$ m$^{-1}$: the chicane's angle for $R_{56} = (1/C - 1)/h$, its $T_{566}$ from tracking, and the RF for the chirp and the curvature $h_2 = -T_{566} h^2/R_{56}$ that cancels it, so the compression is linear: "linearise the compression, not the energy". Each particle's position after the chicane is fitted against its position before: $1/a_1 = 4.001$, and what is left curved, $a_2$, is 0.1 m$^{-1}$, against the predicted $-17.0$ m$^{-1}$ with the energy only linearised.

In [ ]:
__file__ = "37_compression_by_matching.py"  # python and %run set this; a notebook cell does not
"""
37 - Set up JFEL's bunch compression by matching
================================================

Shows: the RF match and a chicane match working together, for a bunch
compressed by a chosen factor. A chirp h (the coefficient of s in the
relative energy, s = c(<t> - t)) through a chicane of R56 (Xsuite's
convention: zeta, delta) takes s to (1 + h R56) s, so a compression factor
C needs

    R56 = (1/C - 1) / h.

The chicane also has T566 (z gains T566 delta^2), so a beam whose energy is
linear in s leaves it curved in s. To compress linearly the RF must give the
beam the curvature that cancels it, h2 = -T566 h^2 / R56: "linearise the
compression, not the energy". The script

1. matches the VBC's angle (`bunch_compressor.angle`) for that R56, from
   the nominal beam;
2. measures the chicane's T566 by tracking through it (a fit of each
   particle's change in z against delta and delta^2);
3. matches the linac RF (L02 and L03, linked by `linac_phase`, and the
   X-band L4H) to the chirp h and the curvature h2, closing the loop by
   tracking as example 36 does;
4. tracks the chicane and checks the compression, against the same RF
   matched to curvature 0 instead.

Everything is tracked in Xsuite with wakes and CSR off, so the only effects
are the RF and the lattice, as the formulas assume.

The compression is read from the particles: each one's s after the chicane
fitted against its s before, s_after = a1 s + a2 s^2 + ... (fifth order, at
the bunch centre, like the match's coefficients). 1/a1 is the compression
and a2 what the chicane leaves curved: R56 h2 + T566 h^2, zero when the RF's
curvature cancels T566. At C = 4 that curvature barely shows in the RMS
length (it adds in quadrature to a much larger linear term); it shows in a2,
which sets how the current profile leans.

Run:    python 06_matching/37_compression_by_matching.py
Tier:   A
Needs:  SIMBA, LAURA, laura-lattices (JFEL) and Xsuite.
Output: output/37_compression_by_matching/: one run per stage.

Units: chirp 1/m, curvature 1/m^2; R56 m, T566 m; sigma_z mm; a2 1/m.

Checks:
- The matched angle gives the target R56 (the tracked fit) to 1e-3.
- The compression is C: 1/a1 to 2e-3, the RMS ratio to 2%.
- With the curvature target, a2 is under 2% of what it is with the energy
  only linearised (measured 0.1 against -17.0 1/m); and that -17.0 is the
  predicted R56 h2 + T566 h^2 to 2%.
"""

from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from jfel import JFEL_DIR, L4H, VBC_END, VBC_START, compressor_def, longitudinal_shape
from scipy.constants import c

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
OUT.mkdir(parents=True, exist_ok=True)
C_TARGET, CHIRP = 4.0, -15.0
R56 = (1 / C_TARGET - 1) / CHIRP
PASSES = 3


def framework_for(name):
    framework = fw.Framework(master_lattice=JFEL_DIR, directory=str(OUT / name), clean=True, verbose=False)
    framework.loadSettings(str(compressor_def(OUT / f"{name}.def", "xsuite")))
    for line in ("Linac", "BC"):
        framework[line].csr_enable = framework[line].lsc_enable = framework[line].wakefield_enable = False
    return framework


def beam(name, element):
    return rbf.load_file(str(OUT / name / f"{element}.openpmd.hdf5"))


def chicane_terms(start, end):
    """R56 and T566 of the tracked chicane: the change in z (ahead positive) fitted against delta and delta^2."""
    delta = start.cp.val / np.mean(start.cp.val) - 1
    dz = (end.z.val - np.mean(end.z.val)) - (start.z.val - np.mean(start.z.val))
    a = np.c_[delta, delta**2, start.x.val, start.cpx.val / start.cpz.val, np.ones_like(delta)]
    r56, t566 = np.linalg.lstsq(a, dz, rcond=None)[0][:2]
    return r56, t566


def sigma_z(b):
    return np.std(c * np.asarray(b.t.val))


def mapping(start, end):
    """
    a1, a2 of s_after = a1 s + a2 s^2 + ..., particle by particle, with
    s = c(<t> - t): the compression (1/a1) and what is left curved (a2), at the
    bunch centre. Fitted to fifth order, as the match's own coefficients are: a
    cubic fit would fold the bunch's fourth order (large over 0.46 mm at X-band)
    into a2.
    """
    s0 = c * (np.mean(start.t.val) - np.asarray(start.t.val))
    s1 = c * (np.mean(end.t.val) - np.asarray(end.t.val))
    coefficients = np.polyfit(s0, s1, 5)[::-1]
    return coefficients[1], coefficients[2]


# 1. The chicane's angle for R56, from the nominal beam
name = "chicane"
framework = framework_for(name)
framework.track(files=["Linac"])
framework["BC"].file_block["match"] = {"variables": ["bunch_compressor.angle"], "targets": {VBC_END: {"R56": R56}}}
framework.track(files=["BC"])
angle = framework["bunch_compressor"].angle
# 2. Its R56 and T566, by tracking
r56, t566 = chicane_terms(beam(name, VBC_START), beam(name, VBC_END))
curvature = -t566 * CHIRP**2 / r56
print(f"target R56 {R56:.6f} m: angle {angle:.6f} rad, tracked R56 {r56:.6f} m, T566 {t566:.6f} m")
print(f"curvature to cancel T566: {curvature:.2f} 1/m^2")


def compress(name, curvature):
    """Match the RF to the chirp and `curvature` (closing the loop), set the angle, track the chicane: the compression."""
    framework = framework_for(name)
    framework.track(files=["Linac"])
    e_target = longitudinal_shape(OUT / name / f"{VBC_START}.openpmd.hdf5")[0]
    e_in = longitudinal_shape(Path(JFEL_DIR) / "Data_Files" / "jfel_injector_S02.openpmd.hdf5")[0]
    target = np.array([CHIRP, curvature, 0.0])
    asked, gain = target.copy(), e_target - e_in
    for _ in range(PASSES):
        framework["Linac"].file_block["longitudinal_match"] = {
            "cavities": "linac", "harmonics": [L4H], "energy_gain": float(gain),
            "chirp": float(asked[0]), "curvature": float(asked[1]), "skewness": float(asked[2]),
        }
        framework.track(files=["Linac"])
        energy, shape = longitudinal_shape(OUT / name / f"{VBC_START}.openpmd.hdf5")
        asked, gain = asked + (target - shape), gain + (e_target - energy)
    framework["bunch_compressor"].set_angle(angle)
    framework.track(files=["BC"])
    before, after = beam(name, VBC_START), beam(name, VBC_END)
    a1, a2 = mapping(before, after)
    print(f"{name:12s} {shape[0]:8.3f} {shape[1]:10.2f} {sigma_z(before) * 1e3:9.4f} {sigma_z(after) * 1e3:9.4f} "
          f"{sigma_z(before) / sigma_z(after):7.3f} {1 / a1:7.3f} {a2:9.3f}")
    return sigma_z(before) / sigma_z(after), a1, a2, shape[1]


print(f"\n{'RF matched':12s} {'chirp':>8s} {'curvature':>10s} {'sigma_z0':>9s} {'sigma_z1':>9s} "
      f"{'C (rms)':>7s} {'1/a1':>7s} {'a2 [1/m]':>9s}")
compression, a1, a2, _ = compress("compensated", curvature)
_, _, a2_plain, h2_plain = compress("linearised", 0.0)
predicted = r56 * h2_plain + t566 * CHIRP**2
print(f"\nleft curved with curvature 0: a2 = {a2_plain:.3f} 1/m, predicted R56 h2 + T566 h^2 = {predicted:.3f} 1/m")

# --- Checks ----------------------------------------------------------------
problems = []
if abs(r56 / R56 - 1) > 1e-3:
    problems.append(f"tracked R56 {r56:.6f}")
if abs(compression / C_TARGET - 1) > 0.02 or abs(1 / a1 / C_TARGET - 1) > 2e-3:
    problems.append(f"compression {compression:.3f} (rms), {1 / a1:.3f} (mapping)")
if abs(a2) > 0.02 * abs(predicted):
    problems.append(f"compensated, still curved: a2 {a2:.3f}")
if abs(a2_plain / predicted - 1) > 0.02:
    problems.append(f"curvature 0: a2 {a2_plain:.3f}, predicted {predicted:.3f}")
assert not problems, "\n".join(problems)
print(f"OK: compressed by {compression:.2f} (target {C_TARGET}) and linearly; with the energy merely "
      f"linearised the chicane's T566 leaves a2 = {a2_plain:.1f} 1/m")